In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup

In [2]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx', 'raw_SDG_2026.xlsx']}

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[2]

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
index_metadata_raw = pd.read_excel(index_metadata_path)
index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    index_metadata_raw['Index_ID'] = '1032'
    index_metadata_raw = index_metadata_raw.merge(country_iso, how='left', left_on='Arab Countries', right_on='country')
    # make first row columns names
    historical_kpi_data_raw.columns = historical_kpi_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

#### Simulator Data

In [3]:
simulator_data = simulator_data_raw.copy()
simulator_data = simulator_data.rename(columns={'Unnamed: 0':'KeyID','Unnamed: 1':'Series name','Unnamed: 2':'Year',
                                        'year':'Year'})
simulator_data.columns = simulator_data.iloc[0]
simulator_data = simulator_data[1:].reset_index(drop=True)
new_names = ['Country', 'ISO', 'Region', 'Sub-region', 'Income']
simulator_data.columns = new_names + list(simulator_data.columns[5:])

# Convert to long format
simulator_data = simulator_data.melt(id_vars=new_names, 
                                     var_name='Series Name', value_name='data_report')      # convert wide to long
cols_to_strip = ['Country', 'ISO', 'Series Name']
simulator_data[cols_to_strip] = simulator_data[cols_to_strip].apply(lambda x: x.str.strip())
simulator_data['data_report'] = pd.to_numeric(simulator_data['data_report'], errors='coerce')
simulator_data = simulator_data.drop(columns=['Region', 'Sub-region', 'Income'])



#### Historical kpi data

In [4]:

# Index data minor reformatting and filtering
historical_kpi_data = historical_kpi_data_raw.copy()
historical_kpi_data = historical_kpi_data.rename(columns={'Unnamed: 0':'KPI ID','Unnamed: 1':'Series Name','Unnamed: 2':'year_historical_automated',
                                        'year':'year_historical_automated', 'Series name':'Series Name', 'KeyID':'KPI ID'})
historical_kpi_data = historical_kpi_data.iloc[1:]
# - Values to change, match simulator values
replacement_dict = {
    'Mean years of schooling': 'Mean Year of Schooling',
    'Expected years of schooling': 'Expected Year of Schooling'
}
historical_kpi_data['Series Name'] = historical_kpi_data['Series Name'].replace(replacement_dict)
# - long format
historical_kpi_data_long = historical_kpi_data.melt(id_vars=['KPI ID','Series Name','year_historical_automated'], 
                                           var_name='ISO', value_name='data_historical_automated')        # convert wide to long
#historical_kpi_data_long = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
# - format values for consistency
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')

series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

In [5]:
simulator_series_names = simulator_data['Series Name'].unique().tolist()
historical_data_series_names = historical_kpi_data_long['Series Name'].unique().tolist()

all_contained = set(historical_data_series_names).issubset(set(simulator_series_names))
missing = set(simulator_series_names) - set(historical_data_series_names)
print(missing)  # empty set if all contained
print(len(missing))

set()
0


#### Reference Tables


In [6]:

# Summary KPI
def find_floor_cap(group, data_col='data_report'):
    counts = group[data_col].value_counts()
    
    min_val = group[data_col].min()
    max_val = group[data_col].max()
    
    min_count = counts.get(min_val, 0)
    max_count = counts.get(max_val, 0)
    # Floor: min value, repeated >4 times, integer
    is_floor = (
        pd.notna(min_val) and
        min_count > 1 and
        float(min_val).is_integer())
    # Cap: max value, repeated >4 times, integer
    is_cap = (
        pd.notna(max_val) and
        max_count > 1 and
        float(max_val).is_integer())
    return pd.Series({
        'likely_cap': max_val if is_cap else np.nan,
        'cap_count': max_count if is_cap else np.nan,
        'likely_floor': min_val if is_floor else np.nan,
        'floor_count': min_count if is_floor else np.nan,
    })
simulator_data_cap = simulator_data.merge(series_to_kpi).copy()
simulator_data_cap = simulator_data_cap.groupby('KPI ID').apply(find_floor_cap).reset_index()
simulator_data_cap['cap_present'] = np.where(~simulator_data_cap['likely_cap'].isna(),1,0)

# Summary KPI/Country data
kpi_country_summary_df = historical_kpi_data_long.groupby(['KPI ID', 'ISO']).agg(
    total_rows=('data_historical_automated', 'size'),
    nonnull_rows=('data_historical_automated', 'count'),).reset_index()
kpi_country_summary_df['nonnull_pct'] = (kpi_country_summary_df['nonnull_rows'] / kpi_country_summary_df['total_rows'] * 100).round(2)
earliest_nonnull = (
    historical_kpi_data_long[historical_kpi_data_long['data_historical_automated'].notna()]
    .groupby(['KPI ID', 'ISO'])['year_historical_automated']
    .max()
    .rename('earliest_year_nonnull')
)
kpi_country_summary_df = kpi_country_summary_df.merge(earliest_nonnull, how='left', on=['KPI ID', 'ISO'])
kpi_country_summary_df['earliest_year_nonnull'] = kpi_country_summary_df['earliest_year_nonnull'].astype('Int64')


# Match Data - to be used for bucketing records
match_data = simulator_data.merge(historical_kpi_data_long, how='left', on=['ISO','Series Name'])        # Attach historical data b
match_data = match_data.merge(kpi_country_summary_df, how='left', on=['KPI ID','ISO'])                   # Include summary data, for reasoning down the line
match_data = match_data[['Country', 'ISO', 'Series Name','KPI ID', 'data_report', 'year_historical_automated',
                         'data_historical_automated', 'total_rows', 'nonnull_rows','nonnull_pct', 'earliest_year_nonnull']]
# - calculated fields
match_data['exact_match'] = match_data['data_report']==match_data['data_historical_automated']
match_data['latest_year'] = match_data['year_historical_automated']==match_data['earliest_year_nonnull']
# - value difference & pct
match_data['value_difference'] = abs(match_data['data_report']-match_data['data_historical_automated'])
denom = match_data['data_report'].replace(0, np.nan)
condition = (match_data['value_difference'] == 0).fillna(False)
match_data['value_difference_percentage'] = np.where(
    condition,
    0,
    (match_data['value_difference'] / denom)
)


# Newest KPI Data for each country
newest_kpi_data = historical_kpi_data_long.copy()
newest_kpi_data = newest_kpi_data.dropna(subset=['data_historical_automated'])                            # drop na
newest_kpi_data = newest_kpi_data[
    newest_kpi_data['year_historical_automated'] ==
    newest_kpi_data.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
newest_kpi_data = newest_kpi_data[['KPI ID', 'ISO', 'year_historical_automated', 'data_historical_automated']]
newest_kpi_data = newest_kpi_data.rename(columns={'year_historical_automated':'most_recent_year', 
                                              'data_historical_automated':'most_recent_data'})


### Recreate metadata

In [30]:

# Start from simulator data
metadata_automated = simulator_data.copy()
#metadata_automated = metadata_automated[metadata_automated['ISO'].isin(escwa_members)]    # Filter for escwa members
print(f"Baseline Simulator data structure: {metadata_automated.shape}")
metadata_automated = metadata_automated.merge(series_to_kpi, how='left',on='Series Name') # Include KPI IDs
metadata_automated = metadata_automated.merge(kpi_country_summary_df, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(simulator_data_cap, how='left', on=['KPI ID'])
metadata_automated = metadata_automated.merge(newest_kpi_data, how='left',on=['KPI ID','ISO']) # Include most recent year
print(f"Structure after additional data: {metadata_automated.shape}")

exact_match = match_data[match_data['exact_match']].copy()
approx_match = match_data[~match_data['exact_match']].copy()
error_tol = 0.01 # 1%

# Col Names
A_exact_year = 'A_1_year'
A_exact_data = 'A_1_data'
A_exact_value_diff = 'A_1_value_diff'
A_exact_value_diff_pct = 'A_1_value_diff_pct'

A_approx_year = 'A_2_year'
A_approx_data = 'A_2_data'
A_approx_value_diff = 'A_2_value_diff'
A_approx_value_diff_pct = 'A_2_value_diff_pct'

B_exact_year = 'B_1_year'
B_exact_data = 'B_1_data'
B_exact_value_diff = 'B_1_value_diff'
B_exact_value_diff_pct = 'B_1_value_diff_pct'

B_approx_year = 'B_2_year'
B_approx_data = 'B_2_data'
B_approx_value_diff = 'B_2_value_diff'
B_approx_value_diff_pct = 'B_2_value_diff_pct'

C_cap_adj_year = 'C_cap_adj_year'
C_cap_adj_data = 'C_cap_adj_data'
C_cap_adj_value_diff = 'C_cap_adj_value_diff'
C_cap_adj_value_diff_pct = 'C_cap_adj_value_diff_pct'


match_cols = ['ISO', 'KPI ID', 'year_historical_automated', 'data_historical_automated','value_difference','value_difference_percentage']

# A - Most Recent Year & Match
# A.1 (Exact Match)
A_exact_match = exact_match[exact_match['latest_year']].copy()
A_exact_match = A_exact_match[match_cols]
A_exact_match = A_exact_match.rename(columns={'year_historical_automated':A_exact_year, 
                                              'data_historical_automated':A_exact_data,
                                              'value_difference':A_exact_value_diff,
                                              'value_difference_percentage':A_exact_value_diff_pct})
# A.2 (Approximate Match)
A_approx_match = approx_match[approx_match['latest_year']].copy()
A_approx_match = A_approx_match[A_approx_match['value_difference_percentage'] <= error_tol]
A_approx_match = A_approx_match[match_cols]
A_approx_match = A_approx_match.rename(columns={'year_historical_automated':A_approx_year, 
                                                'data_historical_automated':A_approx_data,
                                                'value_difference':A_approx_value_diff,
                                                'value_difference_percentage':A_approx_value_diff_pct})
#A_approx_match = A_exact_match[['ISO', 'KPI ID', 'year_historical_automated', 'data_historical_automated']]
#A_exact_match = A_exact_match.rename(columns={'year_historical_automated':'A_1_year', 'data_historical_automated':'A_1_data'})

# B - Older Year & Match
# B.1 (Exact Match)
# - exclude values from A
exclude_keys = pd.concat([A_exact_match[['ISO', 'KPI ID']], A_approx_match[['ISO', 'KPI ID']]]).drop_duplicates()
exclude_index = pd.MultiIndex.from_frame(exclude_keys)
B_exact_match = exact_match[~exact_match['latest_year']].copy()
B_exact_match = B_exact_match[~pd.MultiIndex.from_frame(B_exact_match[['ISO', 'KPI ID']]).isin(exclude_index)]
# - Max year value available
B_exact_match = B_exact_match[
    B_exact_match['year_historical_automated'] ==
    B_exact_match.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
B_exact_match = B_exact_match[match_cols]
B_exact_match = B_exact_match.rename(columns={'year_historical_automated':B_exact_year, 
                                              'data_historical_automated':B_exact_data,
                                              'value_difference':B_exact_value_diff,
                                              'value_difference_percentage':B_exact_value_diff_pct})
# B.2 (Approximate Match)
exclude_keys = pd.concat([A_exact_match[['ISO', 'KPI ID']], A_approx_match[['ISO', 'KPI ID']], B_exact_match[['ISO', 'KPI ID']] ]).drop_duplicates()
exclude_index = pd.MultiIndex.from_frame(exclude_keys)
B_approx_match = approx_match[~approx_match['latest_year']].copy()
B_approx_match = B_approx_match[B_approx_match['value_difference_percentage'] <= error_tol]
B_approx_match = B_approx_match[~pd.MultiIndex.from_frame(B_approx_match[['ISO', 'KPI ID']]).isin(exclude_index)]
# - Max year value available
B_approx_match = B_approx_match[
    B_approx_match['year_historical_automated'] ==
    B_approx_match.groupby(['ISO', 'KPI ID'])['year_historical_automated'].transform('max')]
B_approx_match = B_approx_match[match_cols]
B_approx_match = B_approx_match.rename(columns={'year_historical_automated':B_approx_year, 
                                                'data_historical_automated':B_approx_data,
                                                'value_difference':B_approx_value_diff,
                                                'value_difference_percentage':B_approx_value_diff_pct})

metadata_automated = metadata_automated.merge(A_exact_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(A_approx_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(B_exact_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated = metadata_automated.merge(B_approx_match, how='left', on=['KPI ID', 'ISO'])




metadata_automated['A_exact']= np.where(metadata_automated[A_exact_data].isna(),0,1)
metadata_automated['A_approx']= np.where(metadata_automated[A_approx_data].isna(),0,1)

metadata_automated['A_total']= metadata_automated['A_exact']+metadata_automated['A_approx']

metadata_automated['B_exact']= np.where(metadata_automated[B_exact_data].isna(),0,1)
metadata_automated['B_approx']= np.where(metadata_automated[B_approx_data].isna(),0,1)
metadata_automated['B_total']= metadata_automated['B_exact']+metadata_automated['B_approx']

metadata_automated['A_B_match_total'] = metadata_automated['A_total'] + metadata_automated['B_total']


# C_cap_adjusted match
cap_adjusted_match = metadata_automated.copy()
mask = (
    (cap_adjusted_match['A_B_match_total'] == 0) &                                 # - No A or B match
    (cap_adjusted_match['cap_present'] == 1) &                                     # - cap_present == 1
    (cap_adjusted_match['most_recent_data'].notna()) &                             # - most_recent_data available
    (cap_adjusted_match['most_recent_data'] > cap_adjusted_match['likely_cap']) &  # - most_recent_data > likely_cap
    (cap_adjusted_match['data_report'] == cap_adjusted_match['likely_cap'])        # - data_report == likely_cap
)
cap_adjusted_match = cap_adjusted_match[mask]
cap_adjusted_match[C_cap_adj_year] = cap_adjusted_match['most_recent_year']
cap_adjusted_match[C_cap_adj_data] = cap_adjusted_match['data_report']
cap_adjusted_match[C_cap_adj_value_diff] = 0     # - Assume values is the same as data_report, 0 diff
cap_adjusted_match[C_cap_adj_value_diff_pct] = 0
cap_adjusted_match['C_cap_adjusted_match'] = 1
# - join into df
cap_adjusted_match = cap_adjusted_match[['KPI ID', 'ISO', C_cap_adj_year, C_cap_adj_data, 
                                         C_cap_adj_value_diff, C_cap_adj_value_diff_pct, 'C_cap_adjusted_match']]
metadata_automated = metadata_automated.merge(cap_adjusted_match, how='left', on=['KPI ID','ISO'])
metadata_automated['C_cap_adjusted_match'] = metadata_automated['C_cap_adjusted_match'].fillna(0)
metadata_automated['A_B_C_match_total'] = metadata_automated['A_total'] + metadata_automated['B_total'] + metadata_automated['C_cap_adjusted_match']


#### NO MATCH BUCKETS ####
# Bucket D
# No matches (A,B,or C) & less than or equal to 5 years of historical data
metadata_automated['D_possible_impute'] = (
    (metadata_automated['A_B_C_match_total'] == 0) & # No A/B/C matches
    (metadata_automated['most_recent_data'].notna()) & # Historical data exists
    (metadata_automated['nonnull_rows'] <= 5)        # Less than or equal to 5 yrs historical data
).fillna(False).astype(int)
# Bucket F - No historical data exists
metadata_automated['F_no_historical_data'] = np.where(metadata_automated['most_recent_data'].isna(),1,0)
# Bucket E - everything else
flag_cols_e = ['A_B_C_match_total',
               'D_possible_impute',
               'F_no_historical_data']
metadata_automated['E_material_unexplained_discrepancy'] = (
    (metadata_automated[flag_cols_e] == 0).all(axis=1)
).fillna(False).astype(int)

# Values matches for A, B, C_cap_adjusted
metadata_automated['year_inferred'] = metadata_automated[[A_exact_year, A_approx_year, B_exact_year, B_approx_year, C_cap_adj_year]].bfill(axis=1).iloc[:,0]
metadata_automated['data_inferred'] = metadata_automated[[A_exact_data, A_approx_data, B_exact_data, B_approx_data, C_cap_adj_data]].bfill(axis=1).iloc[:,0]
metadata_automated['year_difference_inferred'] = metadata_automated['most_recent_year'] - metadata_automated['year_inferred']
metadata_automated['value_difference_inferred'] = metadata_automated[[A_exact_value_diff, A_approx_value_diff, 
                                                                      B_exact_value_diff, B_approx_value_diff, 
                                                                      C_cap_adj_value_diff]].bfill(axis=1).iloc[:,0]
metadata_automated['value_difference_percentage_inferred'] = metadata_automated[[A_exact_value_diff_pct, A_approx_value_diff_pct, 
                                                                                 B_exact_value_diff_pct, B_approx_value_diff_pct,
                                                                                 C_cap_adj_value_diff_pct]].bfill(axis=1).iloc[:,0]

# Final Count (A. B, C, D, E, F buckets)
metadata_automated['total_buckets'] = metadata_automated[
    ['A_B_match_total', 'C_cap_adjusted_match','D_possible_impute','E_material_unexplained_discrepancy','F_no_historical_data']
].sum(axis=1)

### Profile Buckets ###
tol_yr_old = 4
# Percent Accurate [A + C]
metadata_automated['Accurate_A_C'] = metadata_automated['A_total']+metadata_automated['C_cap_adjusted_match']
# Percent Old (Within 4 years of most recent data, could be within 4 years of most recent KPI data)
metadata_automated['Old_B'] = np.where(
    (metadata_automated['B_total'] == 1) & (metadata_automated['year_difference_inferred'] <= tol_yr_old), 1, 0)
metadata_automated['Very_Old_B'] = np.where(
    (metadata_automated['B_total'] == 1) & (metadata_automated['year_difference_inferred'] > tol_yr_old), 1, 0)
# Percent Unmatched (D, E, F)
metadata_automated['Unmatched_D_E_F'] = metadata_automated[
    ['D_possible_impute','E_material_unexplained_discrepancy','F_no_historical_data']
].sum(axis=1)
metadata_automated['total_profile'] = metadata_automated[
    ['Accurate_A_C', 'Old_B', 'Very_Old_B', 'Unmatched_D_E_F']
    ].sum(axis=1)




print(f"Structure after data quality info: {metadata_automated.shape}")
print("Rows (first datapoint) should match")
#assert (metadata_automated[bucket_cols].sum(axis=1) == 1).all()
violations = metadata_automated[
    (metadata_automated['B_total'] == 1) &
    (metadata_automated['most_recent_year'].isna() | metadata_automated['year_inferred'].isna())
]

if len(violations) > 0:
    print(f"⚠️ {len(violations)} rows have B_total==1 but are missing most_recent_year or year_inferred")
else:
    print("Confirmed: all B_total==1 rows have both most_recent_year and year_inferred.")

Baseline Simulator data structure: (2895, 4)
Structure after additional data: (2895, 16)
Structure after data quality info: (2895, 59)
Rows (first datapoint) should match
Confirmed: all B_total==1 rows have both most_recent_year and year_inferred.


In [31]:
metadata_automated.to_clipboard()

In [32]:
def kpi_totals(df):
    """Constant-per-KPI fields + total/availability summary. Uses .agg."""
    out = df.groupby('KPI ID').agg(
        likely_cap=('likely_cap', 'first'),
        cap_count=('cap_count', 'first'),
        likely_floor=('likely_floor', 'first'),
        floor_count=('floor_count', 'first'),
        n_datapoints=('KPI ID', 'size'),
        A_exact_sum=('A_exact', 'sum'),
        A_approx_sum=('A_approx', 'sum'),
        B_exact_sum=('B_exact', 'sum'),
        B_approx_sum=('B_approx', 'sum'),
        C_adj_sum=('C_cap_adjusted_match', 'sum'),
        total_sum=('A_B_C_match_total', 'sum'),
    )

    # roll-ups
    out['A_sum'] = out['A_exact_sum'] + out['A_approx_sum']
    out['B_sum'] = out['B_exact_sum'] + out['B_approx_sum']
    out['C_sum'] = out['C_adj_sum']
    out['exact_sum'] = out['A_exact_sum'] + out['B_exact_sum']
    out['approx_sum'] = out['A_approx_sum'] + out['B_approx_sum']
    

    # percentages (denominator = n_datapoints)
    out['pct_A'] = (out['A_sum'] / out['n_datapoints'] * 100).round(1)
    out['pct_B'] = (out['B_sum'] / out['n_datapoints'] * 100).round(1)
    out['pct_C'] = (out['C_sum'] / out['n_datapoints'] * 100).round(1)
    out['pct_exact'] = (out['exact_sum'] / out['n_datapoints'] * 100).round(1)
    out['pct_approx'] = (out['approx_sum'] / out['n_datapoints'] * 100).round(1)
    out['pct_cap_adjusted'] = out['pct_C']
    out['pct_matched'] = (out['total_sum'] / out['n_datapoints'] * 100).round(1)

    return out.reset_index()

def kpi_year_stats(df):
    """Year spread, mode, and distribution per KPI. Needs .apply."""
    def _stats(years):
        years = years.dropna()
        if years.empty:
            return pd.Series({
                'year_min': np.nan, 'year_max': np.nan, 'year_range': np.nan,
                'year_median': np.nan, 'most_common_year': np.nan,
                'most_common_year_pct': np.nan, 'n_distinct_years': 0,
                'year_distribution': {},
            })
        counts = years.value_counts()
        pct = (counts / len(years) * 100).round(1)
        return pd.Series({
            'year_min': years.min(),
            'year_max': years.max(),
            'year_range': years.max() - years.min(),
            'year_median': years.median(),
            'most_common_year': counts.index[0],
            'most_common_year_pct': pct.iloc[0],
            'n_distinct_years': counts.size,
            'year_distribution': pct.sort_index().to_dict(),
        })

    return df.groupby('KPI ID')['year_inferred'].apply(_stats).unstack().reset_index()


In [14]:
summary = kpi_totals(metadata_automated_escwa).merge(kpi_year_stats(metadata_automated_escwa), on='KPI ID', how='left')


In [33]:
# Final column ordering
base_cols=['Country', 'ISO', 'Series Name','KPI ID','data_report', 'total_rows',
          'nonnull_rows', 'nonnull_pct', 'earliest_year_nonnull', 'likely_cap',
          'cap_count', 'likely_floor', 'floor_count', 'cap_present', 'most_recent_year', 'most_recent_data']
inferred_cols=['year_inferred', 'data_inferred','year_difference_inferred', 'value_difference_inferred','value_difference_percentage_inferred']
totals_cols=['A_exact', 'A_approx','A_total', 'B_exact', 'B_approx', 'B_total', 'A_B_match_total', 
             'C_cap_adjusted_match','D_possible_impute','E_material_unexplained_discrepancy','F_no_historical_data', 'total_buckets']
profile_totals_cols = ['Accurate_A_C', 'Old_B', 'Very_Old_B', 'Unmatched_D_E_F', 'total_profile']

# Filter for ESCWA states
metadata_automated_escwa = metadata_automated[metadata_automated['ISO'].isin(escwa_members)].reset_index(drop=True).copy()
metadata_automated_escwa = metadata_automated_escwa[base_cols+inferred_cols+totals_cols+profile_totals_cols]



### Profile Overview

In [34]:
# Table 1 - All Countries

In [36]:
import pandas as pd

def data_quality_summary(metadata_automated, quality_cols):
    """
    Summarize data quality flags across the whole dataset.

    Parameters
    ----------
    metadata_automation : pd.DataFrame
        The dataframe containing the quality flag columns.
    quality_cols : list[str]
        Column names representing the quality categories (values summed per row).

    Returns
    -------
    pd.DataFrame with a row per category (Count, Percentage) plus a Total row.
    """
    totals = metadata_automated[quality_cols].sum()
    total_records = totals.sum()

    if metadata_automated.shape[0] == total_records:
        print(f"All {metadata_automated.shape[0]} records accounted for.")
    else:
        print(f"Record count mismatch {total_records} of {metadata_automated.shape[0]}")

    summary_stats = pd.DataFrame({
        'Category': quality_cols,
        'Count': totals.values,
    })
    summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
    summary_stats.loc[len(summary_stats)] = ['Total', total_records, 100.0]

    return summary_stats


quality_cols = ['Accurate_A_C', 'Old_B', 'Very_Old_B', 'Unmatched_D_E_F']
summary_stats_all = data_quality_summary(metadata_automated, quality_cols)
summary_stats_escwa = data_quality_summary(metadata_automated_escwa, quality_cols)

All 2895 records accounted for.
All 315 records accounted for.


In [51]:
def quality_summary_table(metadata_automated, quality_cols, groupby=None, pct_only=False):
    """
    Build a data quality summary table, either overall or grouped.

    Parameters
    ----------
    metadata_automation : pd.DataFrame
        The dataframe containing the quality flag columns.
    quality_cols : list[str]
        Column names representing the quality categories (summed per row/group).
    groupby : str or list[str], optional
        Column(s) to group by. If None, returns the overall summary
        (with a 'Total' row and 0-100 scale percentages).
        If provided, returns a per-group breakdown (with 0-1 scale '_pct' columns).
    pct_only : bool, default False
        Only applies when `groupby` is provided. If True, restricts the
        output to the groupby column(s) plus the '_pct' columns only
        (drops raw Count and Total columns).

    Returns
    -------
    pd.DataFrame
    """
    if groupby is None:
        # --- overall ----------------------------------------------------
        totals = metadata_automated[quality_cols].sum()
        total_records = totals.sum()

        if metadata_automated.shape[0] == total_records:
            print(f"All {metadata_automated.shape[0]} records accounted for.")
        else:
            print(f"Record count mismatch {total_records} of {metadata_automation.shape[0]}")

        summary_stats = pd.DataFrame({
            'Category': quality_cols,
            'Count': totals.values,
        })
        summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
        summary_stats.loc[len(summary_stats)] = ['Total', total_records, 100.0]

        return summary_stats

    else:
        # --- grouped ------------------------------------------------------
        summary_stats_group = metadata_automated.groupby(groupby)[quality_cols].sum().reset_index()
        summary_stats_group['Total'] = summary_stats_group[quality_cols].sum(axis=1)
        for col in quality_cols:
            summary_stats_group[col + '_pct'] = (summary_stats_group[col] / summary_stats_group['Total']).round(2)

        if pct_only:
            groupby_cols = [groupby] if isinstance(groupby, str) else list(groupby)
            pct_cols = [col + '_pct' for col in quality_cols]
            return summary_stats_group[groupby_cols + pct_cols]

        return summary_stats_group
        
quality_cols = ['Accurate_A_C', 'Old_B', 'Very_Old_B', 'Unmatched_D_E_F']
summary_stats_kpi = quality_summary_table(metadata_automated, quality_cols, groupby='KPI ID', pct_only=True)


In [54]:
index_title = 'Index 9'
title = 'General Overview: Overall Quality of Index'
table_dict = {'All Countries':summary_stats_all, 
              'ESCWA Countries':summary_stats_escwa, 
              f'{index_title} - KPI Quality (All Countries)':summary_stats_kpi}


In [57]:
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def write_stacked_tables_to_excel(
    table_dict: dict,
    output_path: str = "output.xlsx",
    sheet_name: str = "Summary",
    index_title: str = None,
    title: str = None,
    gap_rows: int = 1,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    index: bool = False,
):
    """
    Write a dict of DataFrames stacked vertically onto a single Excel sheet.

    Layout produced:

        (blank row)
        <index_title> | <title>
        (blank row)
        <table label 1>
        <header row>
        <data rows...>
        (blank row(s))
        <table label 2>
        <header row>
        <data rows...>
        ...

    Parameters
    ----------
    table_dict : dict
        Mapping of {table_label: dataframe}, written top-to-bottom in dict order.
        Any number of DataFrames, any shapes -- they need not share columns.
    output_path : str
        Path to save the .xlsx file.
    sheet_name : str
        Name of the single sheet all tables are written to.
    index_title, title : str or None
        Written to columns A and B of the top title row. Skipped if both are None.
    gap_rows : int
        Number of blank rows between the end of one table and the next label.
    autofit_columns : bool
        Resize columns to fit their longest value. Label/title rows are excluded
        from the width calculation so a long label does not blow out column A.
    max_col_width : int
        Cap on auto-fit column width.
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font, used only when header_bg_color is set.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        Add thin borders around each table's header + data cells.
    index : bool
        Whether to write each DataFrame's index as a column.
    """
    safe_name = _sanitize_sheet_name(sheet_name)

    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    label_font = Font(bold=True)
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Rows holding labels/titles -- excluded from the autofit calculation.
    skip_rows = set()

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        # Create the sheet up front so labels can be written before any DataFrame.
        writer.book.create_sheet(safe_name)
        ws = writer.book[safe_name]
        writer.sheets[safe_name] = ws

        current_row = 1  # row 1 left blank

        # --- title row ------------------------------------------------------
        if index_title is not None or title is not None:
            current_row += 1
            if index_title is not None:
                ws.cell(row=current_row, column=1, value=index_title).font = label_font
            if title is not None:
                ws.cell(row=current_row, column=2, value=title).font = label_font
            skip_rows.add(current_row)
            current_row += 1  # blank row after the title

        # --- each table -----------------------------------------------------
        for label, df in table_dict.items():
            current_row += 1
            label_row = current_row
            ws.cell(row=label_row, column=1, value=label).font = label_font
            skip_rows.add(label_row)

            header_row = label_row + 1
            df.to_excel(
                writer,
                sheet_name=safe_name,
                startrow=header_row - 1,  # to_excel is 0-indexed
                index=index,
            )

            n_cols = df.shape[1] + (1 if index else 0)
            last_row = header_row + df.shape[0]

            for col_idx in range(1, n_cols + 1):
                cell = ws.cell(row=header_row, column=col_idx)
                cell.font = header_font
                if header_fill:
                    cell.fill = header_fill
                cell.alignment = header_align
            ws.row_dimensions[header_row].height = 22

            if add_borders:
                for row in ws.iter_rows(min_row=header_row, max_row=last_row,
                                        min_col=1, max_col=n_cols):
                    for cell in row:
                        cell.border = border

            current_row = last_row + gap_rows

        # --- autofit across every table --------------------------------------
        if autofit_columns:
            for col_idx in range(1, ws.max_column + 1):
                col_letter = get_column_letter(col_idx)
                max_len = 0
                for cell in ws[col_letter]:
                    if cell.row in skip_rows:
                        continue
                    if cell.value is not None:
                        max_len = max(max_len, len(str(cell.value)))
                if max_len:
                    ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

    print(f"Saved: {output_path}")


def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]

In [59]:
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def write_stacked_tables_to_excel(
    table_dict: dict,
    output_path: str = "output.xlsx",
    sheet_name: str = "Summary",
    index_title: str = None,
    title: str = None,
    gap_rows: int = 1,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    title_bg_color: str = "4472C4",        # Excel-blue banner; None for no fill
    title_font_color: str = "FFFFFF",
    label_bg_colors: tuple = ("DDEBF7", "F2F2F2"),  # light blue / light grey, cycled
    header_bg_color: str = None,
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    index: bool = False,
):
    """
    Write a dict of DataFrames stacked vertically onto a single Excel sheet.

    Layout produced:

        (blank row)
        <index_title> | <title>          <- filled with title_bg_color
        (blank row)
        <table label 1>                  <- filled with label_bg_colors[0]
        <header row>
        <data rows...>
        (blank row(s))
        <table label 2>                  <- filled with label_bg_colors[1]
        <header row>
        <data rows...>
        ...

    Parameters
    ----------
    table_dict : dict
        Mapping of {table_label: dataframe}, written top-to-bottom in dict order.
        Any number of DataFrames, any shapes -- they need not share columns.
    output_path : str
        Path to save the .xlsx file.
    sheet_name : str
        Name of the single sheet all tables are written to.
    index_title, title : str or None
        Written to columns A and B of the top title row. Skipped if both are None.
    gap_rows : int
        Number of blank rows between the end of one table and the next label.
    autofit_columns : bool
        Resize columns to fit their longest value. Label/title rows are excluded
        from the width calculation so a long label does not blow out column A.
    max_col_width : int
        Cap on auto-fit column width.
    title_bg_color : str or None
        Hex color (no '#') filling the title row. None for no fill.
    title_font_color : str
        Hex color for title text, used only when title_bg_color is set.
    label_bg_colors : tuple[str] or None
        Hex colors cycled across table label rows, so consecutive tables alternate.
        Pass a longer tuple for more than two rotating colors, or None for no fill.
    header_bg_color : str or None
        Hex color for header cell background fill.
    header_font_color : str
        Hex color for header font, used only when header_bg_color is set.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        Add thin borders around each table's header + data cells.
    index : bool
        Whether to write each DataFrame's index as a column.
    """
    safe_name = _sanitize_sheet_name(sheet_name)

    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

    title_font = Font(bold=True, color=title_font_color if title_bg_color else "000000")
    title_fill = PatternFill(
        start_color=title_bg_color, end_color=title_bg_color, fill_type="solid"
    ) if title_bg_color else None

    label_font = Font(bold=True)
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Title banner spans the widest table so it doesn't stop short mid-sheet.
    max_width = max(
        (df.shape[1] + (1 if index else 0)) for df in table_dict.values()
    ) if table_dict else 1

    # Rows holding labels/titles -- excluded from the autofit calculation.
    skip_rows = set()

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        # Create the sheet up front so labels can be written before any DataFrame.
        writer.book.create_sheet(safe_name)
        ws = writer.book[safe_name]
        writer.sheets[safe_name] = ws

        current_row = 1  # row 1 left blank

        # --- title row ------------------------------------------------------
        if index_title is not None or title is not None:
            current_row += 1
            ws.cell(row=current_row, column=1, value=index_title)
            ws.cell(row=current_row, column=2, value=title)
            for col_idx in range(1, max_width + 1):
                cell = ws.cell(row=current_row, column=col_idx)
                cell.font = title_font
                if title_fill:
                    cell.fill = title_fill
            skip_rows.add(current_row)
            current_row += 1  # blank row after the title

        # --- each table -----------------------------------------------------
        for i, (label, df) in enumerate(table_dict.items()):
            current_row += 1
            label_row = current_row
            ws.cell(row=label_row, column=1, value=label).font = label_font
            skip_rows.add(label_row)

            n_cols = df.shape[1] + (1 if index else 0)

            if label_bg_colors:
                color = label_bg_colors[i % len(label_bg_colors)]
                label_fill = PatternFill(start_color=color, end_color=color, fill_type="solid")
                for col_idx in range(1, n_cols + 1):
                    ws.cell(row=label_row, column=col_idx).fill = label_fill

            header_row = label_row + 1
            df.to_excel(
                writer,
                sheet_name=safe_name,
                startrow=header_row - 1,  # to_excel is 0-indexed
                index=index,
            )

            last_row = header_row + df.shape[0]

            for col_idx in range(1, n_cols + 1):
                cell = ws.cell(row=header_row, column=col_idx)
                cell.font = header_font
                if header_fill:
                    cell.fill = header_fill
                cell.alignment = header_align
            ws.row_dimensions[header_row].height = 22

            if add_borders:
                for row in ws.iter_rows(min_row=header_row, max_row=last_row,
                                        min_col=1, max_col=n_cols):
                    for cell in row:
                        cell.border = border

            current_row = last_row + gap_rows

        # --- autofit across every table --------------------------------------
        if autofit_columns:
            for col_idx in range(1, ws.max_column + 1):
                col_letter = get_column_letter(col_idx)
                max_len = 0
                for cell in ws[col_letter]:
                    if cell.row in skip_rows:
                        continue
                    if cell.value is not None:
                        max_len = max(max_len, len(str(cell.value)))
                if max_len:
                    ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

    print(f"Saved: {output_path}")


def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]

In [60]:
file_name = f"Profile_output_template.xlsx"
output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / file_name

index_title = 'Index 9'
title = 'General Overview: Overall Quality of Index'
table_dict = {'All Countries': summary_stats_all,
              'ESCWA Countries': summary_stats_escwa,
              f'{index_title} - KPI Quality (All Countries)': summary_stats_kpi}

write_stacked_tables_to_excel(table_dict,
                              output_path=output_path,
                              index_title=index_title,
                              title=title)

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\Profile_output_template.xlsx


In [44]:

quality_cols = ['A_total',
                'B_total',
                'C_cap_adjusted_match',
                'D_possible_impute',
                'E_material_unexplained_discrepancy',
                'F_no_historical_data']

In [56]:
# Check if data is over the cap if there
metadata_automated_escwa.to_clipboard()

In [40]:
#kpi_totals_summary = kpi_totals(metadata_automated)
summary = kpi_totals(metadata_automated).merge(kpi_year_stats(metadata_automated), on='KPI ID', how='left')
summary_out = summary.copy()
summary_out['C_inaccurate_cap_present']= np.where((~summary_out['likely_cap'].isna()) & (summary_out['pct_matched']<50) ,1,0)
summary_out['C_inaccurate_data_across_multiple_years']= np.where((summary_out['likely_cap'].isna())&(summary_out['n_distinct_years']>=3),1,0)


summary_eratic = summary.copy()
summary_eratic = summary_eratic[summary_eratic['pct_exact'] != 100] # - remove KPIs with perfect matching

# Inaccurate / Cap present
# - exact match

summary_eratic['C_inaccurate_cap_present']= np.where((~summary_eratic['likely_cap'].isna()) & (summary_eratic['pct_matched']<50) ,1,0)
summary_eratic['C_inaccurate_data_across_multiple_years']= np.where((summary_eratic['likely_cap'].isna())&(summary_eratic['n_distinct_years']>=3),1,0)

summary_eratic_merge=summary_eratic[['KPI ID', 'C_inaccurate_cap_present', 'C_inaccurate_data_across_multiple_years']].copy()


In [41]:
summary_out.to_clipboard()

In [33]:
metadata_automated.to_clipboard()

In [315]:

metadata_automated = metadata_automated.merge(summary_eratic_merge, how='left', on='KPI ID')
c_cols = ['C_inaccurate_cap_present', 'C_inaccurate_data_across_multiple_years']
metadata_automated[c_cols] = metadata_automated[c_cols].fillna(0)
mask = (
    (metadata_automated['match_total'] == 1).fillna(False) |
    metadata_automated['most_recent_data'].isna()
)
metadata_automated.loc[mask, c_cols] = 0

# Bucket D
metadata_automated['D_possible_impute'] = (
    (metadata_automated['match_total'] == 0) &
    (metadata_automated['C_inaccurate_cap_present'] == 0) &
    (metadata_automated['C_inaccurate_data_across_multiple_years'] == 0) &
    (metadata_automated['nonnull_rows'] <= 5)
).fillna(False).astype(int)

# Bucekt E
flag_cols_e = ['match_total',
               'C_inaccurate_cap_present',
               'C_inaccurate_data_across_multiple_years',
               'D_possible_impute']

metadata_automated['E_material_unexplained_discrepancy'] = (
    (metadata_automated[flag_cols_e] == 0).all(axis=1)
).fillna(False).astype(int)



print(f"Structure after data quality info: {metadata_automated.shape}")
print("Rows (first datapoint) should match")

Structure after data quality info: (315, 46)
Rows (first datapoint) should match


In [316]:
# Data check
bucket_cols = ['A_total','B_total', 'C_inaccurate_cap_present',
               'C_inaccurate_data_across_multiple_years',
               'D_possible_impute', 'E_material_unexplained_discrepancy']
assert (metadata_automated[bucket_cols].sum(axis=1) == 1).all()

In [45]:
quality_cols = ['A_total',
                'B_total',
                'C_inaccurate_cap_present',
                'C_inaccurate_data_across_multiple_years',
                'D_possible_impute',
                'E_material_unexplained_discrepancy']

# --- overall ----------------------------------------------------------------
totals = metadata_automated[quality_cols].sum()
total_records = totals.sum()

if metadata_automated.shape[0] == total_records:
    print(f"All {metadata_automated.shape[0]} records accounted for.")
else:
    print(f"Record count mismatch {total_records} of {metadata_automated.shape[0]}")

summary_stats = pd.DataFrame({
    'Category': quality_cols,
    'Count': totals.values,
})
summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
summary_stats.loc[len(summary_stats)] = ['Total', total_records, 100.0]

# --- by KPI -----------------------------------------------------------------
summary_stats_kpi = metadata_automated.groupby('KPI ID')[quality_cols].sum().reset_index()
summary_stats_kpi['Total'] = summary_stats_kpi[quality_cols].sum(axis=1)

for col in quality_cols:
    summary_stats_kpi[col + '_pct'] = (summary_stats_kpi[col] / summary_stats_kpi['Total']).round(2)

# --- by country -------------------------------------------------------------
summary_stats_country = metadata_automated.groupby('Country')[quality_cols].sum().reset_index()
summary_stats_country['Total'] = summary_stats_country[quality_cols].sum(axis=1)

for col in quality_cols:
    summary_stats_country[col + '_pct'] = (summary_stats_country[col] / summary_stats_country['Total']).round(2)

KeyError: "['C_inaccurate_cap_present', 'C_inaccurate_data_across_multiple_years'] not in index"

### Output to file

In [318]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
def write_dataframes_to_excel(
    dataframes: dict,
    output_path: str = "output.xlsx",
    freeze_header: bool = True,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,#"4472C4",   # Excel-blue; set None for no fill
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    add_autofilter: bool = True,
    index: bool = False,
):
    """
    Write a dict of DataFrames to one Excel file, one sheet per DataFrame.
 
    Parameters
    ----------
    dataframes : dict
        Mapping of {sheet_name: dataframe}. Sheet names are what you control
        naming for — just use whatever keys you want.
        Example: {"Sales": df_sales, "Inventory": df_inv}
    output_path : str
        Path to save the .xlsx file.
    freeze_header : bool
        If True, freezes the header row so it stays visible when scrolling.
    autofit_columns : bool
        If True, resizes each column to fit its longest value.
    max_col_width : int
        Cap on auto-fit column width (prevents huge columns from long text).
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font color.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        If True, adds thin borders around all cells.
    add_autofilter : bool
        If True, adds filter dropdowns to the header row.
    index : bool
        Whether to write the DataFrame index as a column.
    """
 
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dataframes.items():
            # Excel sheet names max 31 chars, no : \ / ? * [ ]
            safe_name = _sanitize_sheet_name(sheet_name)
            df.to_excel(writer, sheet_name=safe_name, index=index)
 
            worksheet = writer.sheets[safe_name]
            _format_worksheet(
                worksheet,
                df,
                freeze_header=freeze_header,
                autofit_columns=autofit_columns,
                max_col_width=max_col_width,
                header_bg_color=header_bg_color,
                header_font_color=header_font_color,
                header_bold=header_bold,
                add_borders=add_borders,
                add_autofilter=add_autofilter,
                index=index,
            )
 
    print(f"Saved: {output_path}")
 
 
def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]
 
 
def _format_worksheet(
    ws,
    df,
    freeze_header,
    autofit_columns,
    max_col_width,
    header_bg_color,
    header_font_color,
    header_bold,
    add_borders,
    add_autofilter,
    index,
):
    n_cols = df.shape[1] + (1 if index else 0)
    n_rows = df.shape[0] + 1  # +1 for header row
 
    # --- Header styling ---
    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        if header_fill:
            cell.fill = header_fill
        cell.alignment = header_align
 
    # Slightly taller header row to fit wrapped text
    ws.row_dimensions[1].height = 22
 
    # --- Optional thin borders on all used cells ---
    if add_borders:
        thin = Side(style="thin", color="B7B7B7")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=1, max_row=n_rows, min_col=1, max_col=n_cols):
            for cell in row:
                cell.border = border
 
    # --- Freeze header row (and index column if present) ---
    if freeze_header:
        ws.freeze_panes = "B2" if index else "A2"
 
    # --- Autofilter on header row ---
    if add_autofilter:
        ws.auto_filter.ref = ws.dimensions
 
    # --- Autofit column widths based on longest value ---
    if autofit_columns:
        for col_idx in range(1, n_cols + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

In [319]:
file_name = f"Metadata_Automation_{index_name}_V2.xlsx"
output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / file_name

sheets = {
    'Metadata_automated' : metadata_automated,
    'KPI Summary for Buckets' : summary_out,
    'Summary Stats' : summary_stats,
    'Summary Stats by KPI' : summary_stats_kpi,
    'Summary Stats by Country' : summary_stats_country,
    'Raw Data Long format' : historical_kpi_data_long
}

write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
    )

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\Metadata_Automation_index_9_V2.xlsx


In [137]:
# Tier A
# - exact match, most recent year in historical data
# Tier B
# - exact match from previous year
# - match (1-2% error) from previous year
# Tier C (Inaccurate)
# -  no match within (1-2% error) & country has regular data updates
# Tier D (Imputed)
# - nomatch within (1-2% error) & sparse historical data for Country/KPI

exact_match_df
# data_report, KPI, Year, data_historical, most_recent_year (bool if this is the hihest year ISO/KPI), data_match - bool
previous_year_match_df

0       DZA
1       DZA
2       BHR
3       BHR
4       COM
       ... 
6211    SDN
6212    SYR
6213    TUN
6214    ARE
6215    YEM
Name: ISO, Length: 6216, dtype: str

# Most recent historical data

In [271]:
# Composite key
newest_index_data = index_data.copy()

newest_index_data = newest_index_data.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
newest_index_data = newest_index_data.dropna(subset=['data_historical_automated'])                            # drop na
newest_index_data = newest_index_data[newest_index_data['Country'].isin(arab_countries)] # filter for arab countries
# groupby and extract most recent year value
max_year_idx = newest_index_data.groupby(['KeyID', 'Country'])['Year'].idxmax()
newest_index_data = newest_index_data.loc[max_year_idx].reset_index(drop=True)

# Reformat values
newest_index_data['data_historical_automated'] = newest_index_data['data_historical_automated'].astype('Float64')

# Rename composite key (KeyID, Country) ==> (KPI_ID, ISO) to match metadata format
newest_index_data = newest_index_data.rename(columns={'KeyID':'KPI ID', 'Country':'ISO', 'Year':'year_historical_automated'})
newest_index_data['KPI ID'] = newest_index_data['KPI ID'].astype(int).astype(str)

In [272]:
# Composite key
data_long_format = index_data.copy()

data_long_format = data_long_format.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
data_long_format = data_long_format.dropna(subset=['data_historical_automated'])                            # drop na
data_long_format = data_long_format[data_long_format['Country'].isin(arab_countries)]

In [274]:
# Discrepancies between Index and historical data
index_metadata_kpis = index_metadata['KPI ID'].unique().tolist()
historical_index_kpis = newest_index_data['KPI ID'].unique().tolist()
result = list(set(index_metadata_kpis) - set(historical_index_kpis))
result

[]

# Recreate metadata

In [77]:
# Use current file as template
metadata_automated = index_metadata[['Index_ID','KPI name in the report ','Arab Countries','ISO','KPI ID','Year for each country', 'year of the report', 
                                     'data', 'year historical', 'data historical', 'Data availability and treatment',
                                    'Year difference', 'last year available used to compute the index', 'Value difference',
                                    'Accuracy problem']].copy()

# join in automated data pull
metadata_automated = metadata_automated.merge(newest_index_data, how='left', on=['KPI ID', 'ISO'])
# Recreate calculated fields
# G 'Year for each country'
# I 'year of the report'
#metadata_check = index_metadata_check.merge(metadata_recreate[['KPI ID', 'ISO', 'Year', 'data']], how='left', on=['KPI ID','ISO'])

KeyError: "['KPI name in the report '] not in index"

### QA Metrics

In [276]:
data_hist_col = 'data_historical_automated'
year_hist_col = 'year_historical_automated'

metadata_automated['value_difference'] = abs(metadata_automated['data']-metadata_automated[data_hist_col])
metadata_automated['value_difference_percentage'] = np.where(
    (metadata_automated['value_difference'] == 0) & (metadata_automated['data'] == 0),
    0,
    metadata_automated['value_difference'] / metadata_automated['data']
)

metadata_automated['value_equal'] = (
    (metadata_automated['value_difference_percentage'] < 0.01).fillna(False)
)


In [277]:
# Match from previous years

# Value does not match most previous years data
mismatch = metadata_automated[~metadata_automated['value_equal']]

mismatch = mismatch[['ISO', 'KPI ID', 'data']].copy()

# Join in previous data
index_data_long = index_data.copy()
index_data_long = index_data_long.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_previous_year')        # convert wide to long
index_data_long = index_data_long.dropna(subset=['data_historical_previous_year'])                            # drop na
index_data_long = index_data_long[index_data_long['Country'].isin(arab_countries)]
index_data_long = index_data_long.rename(columns={'KeyID':'KPI ID', 'Country':'ISO', 'Year':'year_historical_previous'})
index_data_long['KPI ID'] = index_data_long['KPI ID'].astype(int).astype(str)
index_data_long['year_historical_previous'] = index_data_long['year_historical_previous'].astype(int)

# Merge
mismatch = mismatch.merge(index_data_long, how='left', on=['KPI ID', 'ISO'])
mismatch = mismatch[mismatch['year_historical_previous'] >=2019]
mismatch['value_difference_previous_year'] = abs(mismatch['data']- mismatch['data_historical_previous_year'])
mismatch['value_difference_pct_previous_year'] = mismatch['value_difference_previous_year']/mismatch['data']
mismatch = mismatch[mismatch['value_difference_pct_previous_year']<0.0075] # within 0.75%
# Select based on most recent year
YEAR = 'year_historical_previous'
TOL = 1e-9 # tolerance
mismatch['is_zero'] = mismatch['value_difference_pct_previous_year'].abs() < TOL

ranked = mismatch.sort_values(
    ['ISO', 'KPI ID', 'is_zero', YEAR],
    ascending=[True, True, False, False]
)
ranked['selection_rank'] = ranked.groupby(['ISO', 'KPI ID']).cumcount() + 1
ranked['match_previous_year'] = ranked['selection_rank'] == 1
ranked = ranked.reset_index(drop=True)

previous_year_match = ranked.copy()
previous_year_match = previous_year_match[previous_year_match['match_previous_year']]
previous_year_match = previous_year_match[['ISO', 'KPI ID','year_historical_previous', 'data_historical_previous_year', 'value_difference_previous_year', 'value_difference_pct_previous_year',
                                          'match_previous_year']]

print(metadata_automated.shape)
metadata_automated = metadata_automated.merge(previous_year_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated['year_difference'] = metadata_automated['year_historical_automated']-metadata_automated['year_historical_previous']
metadata_automated['match_previous_year'] =metadata_automated['match_previous_year'].fillna(False)

print(metadata_automated.shape)

(315, 20)
(315, 26)


In [278]:
metadata_automated['Record_Quality_Good'] = np.where(metadata_automated['value_equal'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_Old'] = np.where(metadata_automated['match_previous_year'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_No_Historical_Data_Available']= np.where(metadata_automated['data_historical_automated'].isna(),
                                                    1,
                                                    0)

metadata_automated['Record_Quality_Inaccurate_Imputed'] = np.where( ((metadata_automated['Record_Quality_Good']==0) & (metadata_automated['Record_Quality_Old']==0)) & (metadata_automated['Record_Quality_No_Historical_Data_Available']==0),
                                                                   1,
                                                                   0)
metadata_automated['Record_Quality_Total'] = metadata_automated['Record_Quality_Total'] = metadata_automated['Record_Quality_Good']+metadata_automated['Record_Quality_Old']+metadata_automated['Record_Quality_Inaccurate_Imputed']


### Summary Stats

In [279]:
total_quality_good = sum(metadata_automated['Record_Quality_Good'])
total_quality_old = sum(metadata_automated['Record_Quality_Old'])
total_quality_inaccurate_imputed = sum(metadata_automated['Record_Quality_Inaccurate_Imputed'])
total_quality_no_data_available = sum(metadata_automated['Record_Quality_No_Historical_Data_Available'])

total_records = total_quality_good+total_quality_old+total_quality_inaccurate_imputed+total_quality_no_data_available


if metadata_automated.shape[0] == total_records:
    print(f"All {metadata_automated.shape[0]} records accounted for.")
else:
    print(f"Record count mismatch {total_records} of {metadata_automated.shape[0]}")

summary_stats = pd.DataFrame({
    'Category': ['Good', 'Old', 'Inaccurate_Imputed', 'No_Historical_Data_Available'],
    'Count': [total_quality_good, total_quality_old, total_quality_inaccurate_imputed, total_quality_no_data_available],
})
summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
# Add a totals row
summary_stats.loc['Total'] = ['Total', total_records, 100.0]

All 315 records accounted for.


### Summary Stats KPI

In [280]:
summary_stats_kpi = metadata_automated.groupby(['KPI ID'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']
].sum().reset_index()

summary_stats_kpi['Total'] = (
    summary_stats_kpi['Record_Quality_Good']
    + summary_stats_kpi['Record_Quality_Old']
    + summary_stats_kpi['Record_Quality_Inaccurate_Imputed']
    + summary_stats_kpi['Record_Quality_No_Historical_Data_Available']
)
# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']:
    summary_stats_kpi[col + '_pct'] = (summary_stats_kpi[col] / summary_stats_kpi['Total']).round(2)

### Summary Stats Country

In [281]:
summary_stats_country = metadata_automated.groupby(['Arab Countries'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']
].sum().reset_index()

summary_stats_country['Total'] = (
    summary_stats_country['Record_Quality_Good']
    + summary_stats_country['Record_Quality_Old']
    + summary_stats_country['Record_Quality_Inaccurate_Imputed']
    + summary_stats_country['Record_Quality_No_Historical_Data_Available']
)

# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']:
    summary_stats_country[col + '_pct'] = (summary_stats_country[col] / summary_stats_country['Total']).round(2)

In [282]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
def write_dataframes_to_excel(
    dataframes: dict,
    output_path: str = "output.xlsx",
    freeze_header: bool = True,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,#"4472C4",   # Excel-blue; set None for no fill
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    add_autofilter: bool = True,
    index: bool = False,
):
    """
    Write a dict of DataFrames to one Excel file, one sheet per DataFrame.
 
    Parameters
    ----------
    dataframes : dict
        Mapping of {sheet_name: dataframe}. Sheet names are what you control
        naming for — just use whatever keys you want.
        Example: {"Sales": df_sales, "Inventory": df_inv}
    output_path : str
        Path to save the .xlsx file.
    freeze_header : bool
        If True, freezes the header row so it stays visible when scrolling.
    autofit_columns : bool
        If True, resizes each column to fit its longest value.
    max_col_width : int
        Cap on auto-fit column width (prevents huge columns from long text).
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font color.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        If True, adds thin borders around all cells.
    add_autofilter : bool
        If True, adds filter dropdowns to the header row.
    index : bool
        Whether to write the DataFrame index as a column.
    """
 
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dataframes.items():
            # Excel sheet names max 31 chars, no : \ / ? * [ ]
            safe_name = _sanitize_sheet_name(sheet_name)
            df.to_excel(writer, sheet_name=safe_name, index=index)
 
            worksheet = writer.sheets[safe_name]
            _format_worksheet(
                worksheet,
                df,
                freeze_header=freeze_header,
                autofit_columns=autofit_columns,
                max_col_width=max_col_width,
                header_bg_color=header_bg_color,
                header_font_color=header_font_color,
                header_bold=header_bold,
                add_borders=add_borders,
                add_autofilter=add_autofilter,
                index=index,
            )
 
    print(f"Saved: {output_path}")
 
 
def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]
 
 
def _format_worksheet(
    ws,
    df,
    freeze_header,
    autofit_columns,
    max_col_width,
    header_bg_color,
    header_font_color,
    header_bold,
    add_borders,
    add_autofilter,
    index,
):
    n_cols = df.shape[1] + (1 if index else 0)
    n_rows = df.shape[0] + 1  # +1 for header row
 
    # --- Header styling ---
    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        if header_fill:
            cell.fill = header_fill
        cell.alignment = header_align
 
    # Slightly taller header row to fit wrapped text
    ws.row_dimensions[1].height = 22
 
    # --- Optional thin borders on all used cells ---
    if add_borders:
        thin = Side(style="thin", color="B7B7B7")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=1, max_row=n_rows, min_col=1, max_col=n_cols):
            for cell in row:
                cell.border = border
 
    # --- Freeze header row (and index column if present) ---
    if freeze_header:
        ws.freeze_panes = "B2" if index else "A2"
 
    # --- Autofilter on header row ---
    if add_autofilter:
        ws.auto_filter.ref = ws.dimensions
 
    # --- Autofit column widths based on longest value ---
    if autofit_columns:
        for col_idx in range(1, n_cols + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

In [283]:

file_name = f"Metadata_Automation_{index_name}_V2.xlsx"
output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / file_name

sheets = {
    'Metadata_automated' : metadata_automated,
    'Summary Stats' : summary_stats,
    'Summary Stats by KPI' : summary_stats_kpi,
    'Summary Stats by Country' : summary_stats_country,
    'Raw Data Long format' : data_long_format
}

write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
    )

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\Metadata_Automation_index_9_V2.xlsx
